In [12]:
import json
import os
from pathlib import Path
import re
from typing import Iterable, List, Tuple

In [13]:
SENTENCE_SPLIT_RE = re.compile(r"\.")

# Hardcoded paths/settings
INPUT_ROOT = Path(r"E:\Dataset\Romanian-BabyLM")
OUTPUT_DIR = Path(r"E:\Dataset\pre-training")
MAX_SENTENCES_PER_PART = 200000
MAX_FILES = None

In [14]:
def iter_text_files(root: Path) -> Iterable[Path]:
    for path in root.rglob("*.txt"):
        if path.is_file():
            yield path

In [15]:
def split_sentences_with_buffer(text: str, buffer: str) -> Tuple[List[str], str]:
    normalized = text.replace("\r", " ").replace("\n", " ")
    cleaned = " ".join(normalized.strip().split())
    if buffer:
        cleaned = f"{buffer} {cleaned}" if cleaned else buffer
    if not cleaned:
        return [], ""
    parts = cleaned.split(".")
    complete = [p.strip() + "." for p in parts[:-1] if p.strip()]
    remainder = parts[-1].strip()
    return complete, remainder

In [16]:
def open_new_part(out_dir: Path, part_index: int):
    out_dir.mkdir(parents=True, exist_ok=True)
    filename = f"part{part_index:04d}.json"
    handle = (out_dir / filename).open("w", encoding="utf-8")
    handle.write("[")
    return handle

In [17]:
def close_part(handle, wrote_any: bool):
    handle.write("]")
    handle.close()
    return False

In [18]:
def write_record(handle, record: dict, wrote_any: bool) -> bool:
    if wrote_any:
        handle.write(",")
    handle.write(json.dumps(record, ensure_ascii=False))
    return True

In [19]:
def convert(
    input_root: Path,
    output_dir: Path,
    max_sentences_per_part: int,
    max_files: int | None,
) -> Tuple[int, int]:
    part_index = 1
    sentence_count = 0
    file_count = 0

    out_file = open_new_part(output_dir, part_index)
    wrote_any = False

    try:
        for file_path in iter_text_files(input_root):
            file_count += 1
            if max_files is not None and file_count > max_files:
                break

            print(f"Processed file: {file_path}")
            buffer = ""
            with file_path.open("r", encoding="utf-8", errors="replace") as handle:
                for line in handle:
                    sentences, buffer = split_sentences_with_buffer(line, buffer)
                    for sent in sentences:
                        record = {"sentence": sent}
                        wrote_any = write_record(out_file, record, wrote_any)
                        sentence_count += 1

                        if sentence_count % max_sentences_per_part == 0:
                            wrote_any = close_part(out_file, wrote_any)
                            part_index += 1
                            out_file = open_new_part(output_dir, part_index)

            # Drop trailing buffer without a closing dot.
            buffer = ""
    finally:
        close_part(out_file, wrote_any)

    return file_count, sentence_count

In [20]:
if MAX_SENTENCES_PER_PART <= 0:
    raise SystemExit("MAX_SENTENCES_PER_PART must be > 0")

file_count, sentence_count = convert(
    input_root=INPUT_ROOT,
    output_dir=OUTPUT_DIR,
    max_sentences_per_part=MAX_SENTENCES_PER_PART,
    max_files=MAX_FILES,
)

print(f"Processed {file_count} file(s), wrote {sentence_count} sentence(s).")

Processed file: E:\Dataset\Romanian-BabyLM\Basme_Fairy Tales\Ion Creangă _Amintiri din Copilarie.txt
Processed file: E:\Dataset\Romanian-BabyLM\Basme_Fairy Tales\Ion Creangă_Danilă Prepeleac.txt
Processed file: E:\Dataset\Romanian-BabyLM\Basme_Fairy Tales\Ion Creangă_Fata Babei şi Fata Moşneagului.txt
Processed file: E:\Dataset\Romanian-BabyLM\Basme_Fairy Tales\Ion Creangă_Harap Alb.txt
Processed file: E:\Dataset\Romanian-BabyLM\Basme_Fairy Tales\Ion Creangă_Ivan Turbincă.txt
Processed file: E:\Dataset\Romanian-BabyLM\Basme_Fairy Tales\Ion Creangă_Ursul Păcălit de Vulpe.txt
Processed file: E:\Dataset\Romanian-BabyLM\Basme_Fairy Tales\Mihai Eminescu_BORTA-VÂNTULUI.txt
Processed file: E:\Dataset\Romanian-BabyLM\Basme_Fairy Tales\Mihai Eminescu_Călin Nebubul.txt
Processed file: E:\Dataset\Romanian-BabyLM\Basme_Fairy Tales\Mihai Eminescu_Fata-n grădina de aur.txt
Processed file: E:\Dataset\Romanian-BabyLM\Basme_Fairy Tales\Mihai Eminescu_Făt Frumos din Lacrimă.txt
Processe